# Definição das informações do histórico dos jogadores

## Objetivo

Definir quais informações das partidas anteriores de um jogador serão usadas pelo modelo para prever a quantidade de kills (`qtKill`) na próxima partida, e quantas partidas anteriores devem ser consideradas.

## Regra principal

Para prever uma partida, só podem ser usadas informações das **partidas anteriores** do jogador. Nenhum dado da própria partida prevista (dano, headshots, vitória, rounds) pode entrar, pois isso causaria vazamento de dados. Por isso, os registros são ordenados por `idPlayer` e `dtCreatedAt`, e cada informação é calculada com `shift(1)`, que exclui a partida atual.

## Informações do histórico definidas

1. Kills da última partida
2. Média de kills das últimas 5 partidas
3. Média de kills das últimas 20 partidas
4. Média de kills de todo o histórico
5. Desvio-padrão das kills recentes
6. Média de dano (`vlDamage`)
7. Média de headshots (`qtHitHeadshot`)
8. Média de tiros por kill (`qtShots / qtKill`)
9. Taxa de vitórias (`flWinner`)
10. Média de rounds jogados (`qtRoundsPlayed`)
11. Média de kills por round
12. Quantidade de partidas anteriores
13. Nível atual do jogador (`vlLevel`)
14. Média de kills do jogador no mapa da partida (`descMapName`)

**Ficam de fora:** dados da própria partida prevista, as colunas `qtHit*` por região do corpo (redundantes com precisão e headshots), e `idPlayer` e `idRoom` (o modelo decoraria o jogador).

## Quantas partidas anteriores usar

Foram comparadas as médias de kills das últimas 1, 3, 5, 10, 20 e 50 partidas e do histórico completo. Cada uma foi usada como previsão das kills da partida seguinte, sem modelo, e o erro foi medido nas mesmas 172.863 linhas (jogadores com pelo menos 5 partidas anteriores).

| Janela | MAE | RMSE |
|---|---|---|
| Última partida | 7,377 | 9,490 |
| Últimas 3 | 6,091 | 7,813 |
| Últimas 5 | 5,782 | 7,436 |
| Últimas 10 | 5,540 | 7,143 |
| Últimas 20 | 5,434 | 7,015 |
| Últimas 50 | 5,387 | 6,960 |
| Histórico completo | 5,385 | 6,958 |

Referência: prever sempre a média geral de kills tem MAE de 5,812.

### O que são MAE e RMSE

Ambas medem o **erro da previsão**, ou seja, a distância entre as kills previstas e as kills reais da partida. Quanto menor, melhor.

**MAE (Mean Absolute Error, erro absoluto médio):** a média da diferença entre o valor previsto e o real, ignorando o sinal.

MAE = média de |kills reais − kills previstas|

Exemplo: um MAE de 5,4 significa que, em média, a previsão erra por cerca de 5,4 kills, para mais ou para menos. É fácil de interpretar porque está na mesma unidade da variável (kills).

**RMSE (Root Mean Squared Error, raiz do erro quadrático médio):** eleva cada erro ao quadrado, tira a média e extrai a raiz.

RMSE = raiz da média de (kills reais − kills previstas)²

Como os erros são elevados ao quadrado, **erros grandes pesam mais**. Por isso o RMSE é sempre maior ou igual ao MAE. Se o RMSE for bem maior que o MAE, há previsões com erros muito grandes em alguns casos.

**Como usamos aqui:** o MAE é a medida principal, por ser direta. O RMSE serve para conferir se as previsões têm erros grandes isolados. Nas duas, o menor valor indica a melhor janela, e ambas apontam a mesma tendência: o erro cai até as 20 partidas e depois quase não muda.


### Conclusões

1. A última partida sozinha erra mais que a média geral (7,377 contra 5,812): uma partida é muito ruidosa.
2. O erro cai conforme a janela aumenta, mas o ganho diminui: de 10 para 20 partidas o MAE cai 0,106, de 20 para 50 cai 0,047, e de 50 para o histórico completo, 0,002.
3. As janelas de 20 partidas e do histórico completo ficam próximas do menor erro observado.

### Decisão

- **Janela principal:** últimas 20 partidas e histórico completo.
- **Fase recente:** últimas 5 partidas, como sinal complementar.
- **Mínimo para prever:** 5 partidas anteriores. Jogadores com menos que isso não entram na previsão.

Esta comparação avalia cada janela isolada. O efeito de combinar várias janelas como features do modelo será verificado na etapa de treinamento.


In [20]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/tb_lobby_stats_player.csv")

In [21]:
## 2. Ordenação por tempo
# Ordena por jogador e data, para que o "histórico" respeite a ordem real das partidas.

df['dtCreatedAt'] = pd.to_datetime(df['dtCreatedAt'])
df = df.sort_values(['idPlayer', 'dtCreatedAt', 'idLobbyGame']).reset_index(drop=True)

In [22]:
## 3. Partidas anteriores
# Cria `qtdAnteriores`: quantas partidas o jogador já tinha jogado antes de cada linha.


df['qtdAnteriores'] = df.groupby('idPlayer').cumcount()

In [23]:
## 4. Médias de kills por janela
# Calcula a média de kills das últimas 1, 3, 5, 10, 20 e 50 partidas e do histórico completo, sem incluir a partida atual (`shift(1)`), para evitar vazamento.

JANELAS = [1, 3, 5, 10, 20, 50]
kills = df.groupby('idPlayer')['qtKill']

for j in JANELAS:
    # shift(1) tira a partida atual; rolling(j) pega as j anteriores
    df[f'mediaKills_{j}'] = kills.transform(
        lambda s: s.shift(1).rolling(j, min_periods=1).mean()
    )

# histórico completo
df['mediaKills_hist'] = kills.transform(lambda s: s.shift(1).expanding().mean())



In [24]:
## 5. Erro de cada janela
# Usa cada média como previsão das kills seguintes e mede o erro (MAE e RMSE), só em jogadores com pelo menos 5 partidas anteriores. Compara com a referência de prever sempre a média geral.

colunas = [f'mediaKills_{j}' for j in JANELAS] + ['mediaKills_hist']

# Só linhas com pelo menos 5 partidas anteriores (a regra do mínimo)
aptos = df[df['qtdAnteriores'] >= 5]

resultado = pd.DataFrame({
    'MAE':  [(aptos['qtKill'] - aptos[c]).abs().mean() for c in colunas],
    'RMSE': [np.sqrt(((aptos['qtKill'] - aptos[c]) ** 2).mean()) for c in colunas],
}, index=colunas)

# Referência: prever sempre a média geral de kills
media_global = aptos['qtKill'].mean()

mae_ref = (aptos['qtKill'] - media_global).abs().mean()

In [25]:
## 6. Tabela comparativa
# Reúne MAE, RMSE e o ganho de cada janela em relação à média geral e à janela anterior.


nomes = ['Última partida', 'Últimas 3', 'Últimas 5', 'Últimas 10',
         'Últimas 20', 'Últimas 50', 'Histórico completo']

tabela_janelas = pd.DataFrame({
    'Janela': nomes,
    'MAE': resultado['MAE'].values,
    'RMSE': resultado['RMSE'].values,
})

# Quanto cada janela melhora em relação a prever sempre a média geral
tabela_janelas['Ganho vs média geral (%)'] = (
    (mae_ref - tabela_janelas['MAE']) / mae_ref * 100
).round(2)

# Quanto o MAE caiu em relação à janela da linha de cima
tabela_janelas['Ganho vs janela anterior'] = (-tabela_janelas['MAE'].diff()).round(3)

tabela_janelas[['MAE', 'RMSE']] = tabela_janelas[['MAE', 'RMSE']].round(3)

display(tabela_janelas)


,Janela,MAE,RMSE,Ganho vs média geral (%),Ganho vs janela anterior
0,Última partida,7.377,9.490,-26.93,NaN
1,Últimas 3,6.091,7.813,-4.80,1.286
2,Últimas 5,5.782,7.436,0.52,0.309
3,Últimas 10,5.540,7.143,4.68,0.241
4,Últimas 20,5.434,7.015,6.51,0.106
5,Últimas 50,5.387,6.960,7.32,0.047
6,Histórico completo,5.385,6.958,7.35,0.002


**Leitura:** o "ganho vs média geral" mede quanto o erro diminui em relação a prever sempre a média geral de kills (MAE de 5,812). O "ganho vs janela anterior" mostra quantas kills de erro se reduzem ao passar para a janela seguinte.

A última partida sozinha erra 26,9% mais que a média geral, e as últimas 3 erram 4,8% mais: poucas partidas geram médias instáveis. A partir de 5 partidas a janela passa a superar a referência, e o erro continua caindo até as últimas 20 partidas (ganho de 6,5%). Depois disso o ganho praticamente estagna: de 20 para 50 partidas o MAE cai só 0,047, e de 50 para o histórico completo, 0,002. Por isso a janela principal definida foi a de 20 partidas, junto com o histórico completo.

ganho = (erro da referência − erro da janela) / erro da referência × 100


In [26]:
## 7. Janelas das demais informações
# Compara as janelas de 5, 20 e do histórico completo para dano, headshots, rounds e vitórias. Como essas médias não são previsões de kills, a medida usada é a correlação entre a média histórica e as kills da partida (`qtKill`): quanto maior o valor absoluto, mais informativa é a janela.


def media_historica(df, coluna, janela=None):
    """Média da coluna nas partidas anteriores do jogador (a atual não entra)."""
    g = df.groupby('idPlayer')[coluna]
    if janela is None:
        return g.transform(lambda s: s.shift(1).expanding().mean())
    return g.transform(lambda s: s.shift(1).rolling(janela, min_periods=1).mean())


In [27]:
ordem = ['qtKill', 'vlDamage', 'qtHitHeadshot', 'qtRoundsPlayed', 'flWinner']

nomes_variaveis = {
    'qtKill': 'Kills',
    'vlDamage': 'Dano',
    'qtHitHeadshot': 'Headshots',
    'qtRoundsPlayed': 'Rounds jogados',
    'flWinner': 'Taxa de vitórias',
}

tabela_corr = (
    pd.DataFrame(linhas)
    .pivot(index='variavel', columns='janela', values='correlacao')[['5', '20', 'hist']]
    .loc[ordem]
    .rename(index=nomes_variaveis,
            columns={'5': 'Últimas 5', '20': 'Últimas 20', 'hist': 'Histórico completo'})
    .rename_axis(index='Média histórica de', columns='Janela')
    .round(3)
)

display(tabela_corr)



Janela,Últimas 5,Últimas 20,Histórico completo
Média histórica de,,,
Kills,0.299,0.358,0.367
Dano,0.293,0.352,0.361
Headshots,0.252,0.287,0.291
Rounds jogados,0.053,0.084,0.095
Taxa de vitórias,0.040,0.067,0.100


**Leitura:** as médias históricas de kills (0,37) e de dano (0,36) são as que mais se relacionam com as kills da partida seguinte, seguidas por headshots (0,29). Rounds jogados e taxa de vitórias têm correlação baixa (abaixo de 0,10). Em todas as variáveis a correlação aumenta da janela de 5 partidas para a de 20 e quase não muda para o histórico completo, o que confirma a escolha de 20 partidas e histórico completo.
